# Required libraries
We will use minsearch and Qdrant. Make sure you have the most up-to-date versions:

```bash
pip install -U minsearch qdrant_client
```
minsearch should be at least 0.0.4.

In [9]:
import minsearch  
minsearch.__version__  # Ensure you have the correct version installed

'0.0.4'

# Evaluation Data
For this homework, we will use the same dataset we generated in the videos.

Let's get them:

In [3]:
import requests
import pandas as pd

url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient='records')

Here, documents contains the documents from the FAQ database with unique IDs, and ground_truth contains generated question-answer pairs.

Also, we will need the code for evaluating retrieval:

In [2]:
from tqdm.auto import tqdm

def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

# Q1. Minsearch text

Now let's evaluate our usual minsearch approach, indexing documents with:
```python 
text_fields=["question", "section", "text"],
keyword_fields=["course", "id"]
```
but tweak the parameters for search. Let's use the following boosting params:
```python
boost = {'question': 1.5, 'section': 0.1}
```
What's the hitrate for this approach?

In [26]:
import minsearch 
minsearch_index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course", "id"]
)
minsearch_index.fit(documents)
boost = {'question': 1.5, 'section': 0.1}


def minsearch_search(q):
    results = minsearch_index.search(
        query=q['question'],
        filter_dict={'course': q['course']},
        boost_dict=boost,
        num_results=5,
    )
    return results



In [27]:

evaluate(ground_truth, minsearch_search)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.848714069591528, 'mrr': 0.7288235717887772}

# Embeddings
The latest version of minsearch also supports vector search. We will use it:


In [28]:

from minsearch import VectorSearch

We will also use TF-IDF and Singular Value Decomposition to create embeddings from texts. You can refer to our "Create Your Own Search Engine" workshop if you want to know more about it.

In [29]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline


Let's create embeddings for the "question" field:

In [30]:
texts = []

for doc in documents:
    t = doc['question']
    texts.append(t)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

# Q2. Vector search for question
Now let's index these embeddings with minsearch:

In [31]:
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)

In [32]:
def minsearch_vector_search(q):
    query_vector = pipeline.transform([q['question']])
    results = vindex.search(query_vector, filter_dict={'course': q['course']}, num_results=5)
    return results

Evaluate this seach method. What's MRR for it?

In [33]:
evaluate(ground_truth, minsearch_vector_search)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.48173762697212014, 'mrr': 0.3572833369353793}

# Q3. Vector search for question and answer
We only used question in Q2. We can use both question and answer:


In [38]:
texts_q3 = []

for doc in documents:
    t = doc['question'] + ' ' + doc['text']
    texts_q3.append(t)

pipeline_q3 = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X_q3 = pipeline_q3.fit_transform(texts_q3)

vindex_q3 = VectorSearch(keyword_fields={'course'})
vindex_q3.fit(X_q3, documents)
def minsearch_vector_search_q3(q):
    query_vector = pipeline_q3.transform([q['question']])
    results = vindex_q3.search(query_vector, filter_dict={'course': q['course']}, num_results=5)
    return results


Using the same pipeline (min_df=3 for TF-IDF vectorizer and n_components=128` for SVD), evaluate the performance of this approach

In [39]:
evaluate(ground_truth,minsearch_vector_search_q3)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.8210503566025502, 'mrr': 0.6717347453353508}

# Q4. Qdrant
Now let's evaluate the following settings in Qdrant:
```python
text = doc['question'] + ' ' + doc['text']
model_handle = "jinaai/jina-embeddings-v2-small-en"
limit = 5
```
What's the MRR?

In [54]:
# Note: this will need the qdrant server running locally
# make sure to run docker compose up in the 03-evaluation directory.
from qdrant_client import QdrantClient, models 
from fastembed import TextEmbedding 
import uuid
qclient = QdrantClient(url='http://localhost:6333')
model_handle = "jinaai/jina-embeddings-v2-small-en"
limit = 5
EMBEDDING_DIMENSIONALITY=512
collection_name = 'zoomcamp_questions'
points = []
for doc in documents: 
    text = doc['question'] + ' ' + doc['text']
    point = models.PointStruct(
        # ID needs to be a uuid, so we can use uuid5 to generate a consistent ID based on the document ID
        # NOTE I just used some namespace here. I just selected one instead of making my own
        id=str(uuid.uuid5(namespace=uuid.NAMESPACE_OID, name=doc['id'])),
        #id=doc['id'],
        vector=models.Document(text=text, model=model_handle),
        payload={
            "course": doc['course'],
            "question": doc['question'],
            "text": doc['text'],
            "id": doc['id'],
        }
    )
    points.append(point)

if qclient.collection_exists(collection_name):
    qclient.delete_collection(collection_name)

qclient.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)
qclient.create_payload_index(
    collection_name=collection_name,
    field_name='course',
    field_schema='keyword',
)
qclient.upsert(
    collection_name='zoomcamp_questions',
    points=points,
)

def qdrant_search(q):
    results = qclient.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=q['question'],
            model=model_handle,
        ),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key='course',
                    match=models.MatchValue(value=q['course'])
                )
            ]
        ),
        limit=limit,
        with_payload=True,
    )
    return [ p.payload for p in results.points ] 

In [55]:
evaluate(ground_truth, qdrant_search)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.9299762264966501, 'mrr': 0.8513039406382837}

# Q5. Cosine simiarity
In the second part of the module, we looked at evaluating the entire RAG approach. In particular, we looked at comparing the answer generated by our system with the actual answer from the FAQ.

One of the ways of doing it is using the cosine similarity. Let's see how to calculate it.

Cosine similarity is a dot product between two normalized vectors. In geometrical sense, it's the cosine of the angle between the vectors. Look up "cosine similarity geometry" if you want to learn more about it.

For us, it means that we need two things:

First, we normalize each of the vectors
Then, compute the dot product

In [128]:
import numpy as np
def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)

results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)

pipeline_q5 = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
pipeline_q5.fit(df_results.answer_llm + ' ' + df_results.answer_orig + ' ' + df_results.question)

df_results['v_llm_list'] = pipeline_q5.transform(df_results.answer_llm).tolist()
df_results['v_llm'] = df_results.apply(lambda row: np.array(row['v_llm_list']), axis=1)
df_results['v_orig_list'] = pipeline_q5.transform(df_results.answer_orig).tolist()
df_results['v_orig'] = df_results.apply(lambda row: np.array(row['v_orig_list']), axis=1)
df_results['cos_answers'] = df_results.apply(lambda row: cosine(row['v_llm'], row['v_orig']), axis=1) 
df_results['cos_answers'].mean()

np.float64(0.8415841233490402)

# Q6. Rouge
And alternative way to see how two texts are similar is ROUGE.

This is a set of metrics that compares two answers based on the overlap of n-grams, word sequences, and word pairs.

It can give a more nuanced view of text similarity than just cosine similarity alone.

We don't need to implement it ourselves, there's a python package for it:

```bash
pip install rouge
```

Let's compute the ROUGE score between the answers at the index 10 of our dataframe (doc_id=5170565b)

```python
from rouge import Rouge
rouge_scorer = Rouge()

r = df_results.iloc[10]
scores = rouge_scorer.get_scores(r.answer_llm, r.answer_orig)[0]
scores
```
There are three scores: rouge-1, rouge-2 and rouge-l, and precision, recall and F1 score for each.

rouge-1 - the overlap of unigrams,
rouge-2 - bigrams,
rouge-l - the longest common subsequence
For the 10th document, Rouge-1 F1 score is 0.45

Let's compute it for the pairs in the entire dataframe. What's the average Rouge-1 F1?